# 06 - Ciclo de Vida Processual STJ

Objetivo: construir uma camada analitica por processo, separando claramente:

- origem do processo no numero CNJ;
- entrada/distribuicao no STJ;
- movimentacoes conhecidas via DataJud/API;
- primeira aparicao no corpus de integras do STJ;
- contagem de documentos/textos por processo.

Este notebook nao tenta recuperar texto integral. Ele prepara a espinha dorsal (`process_spine`) para que o notebook seguinte possa anexar documentos e textos sem duplicar processos.

## 1. Ambiente e parametros

No Colab, mantenha os dados grandes no Drive. Localmente, o notebook usa `data/` na raiz do repositorio.

In [ ]:
# Rode esta celula no Colab se necessario.
# !pip install -r requirements.txt

In [1]:
from __future__ import annotations

import json
import re
import sys
import zipfile
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)

print('Imports loaded.')

Imports loaded.


In [2]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'notebooks').exists() and (candidate / 'src').exists():
            return candidate
        if candidate.name == 'datajud_probe':
            return candidate
    return start

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT = find_project_root(Path.cwd())

if IN_COLAB:
    MOUNT_POINT = Path('/content/drive')
    if (MOUNT_POINT / 'MyDrive').exists():
        print('Google Drive ja esta montado.')
    else:
        drive.mount(str(MOUNT_POINT), force_remount=True)
    DATA_ROOT = MOUNT_POINT / 'MyDrive/Mestrado/2026/llms/data'
else:
    DATA_ROOT = PROJECT_ROOT / 'data'

RAW_DATA = DATA_ROOT / 'raw'
RAW_STJ = RAW_DATA / 'stj_integras'
METADATA_RAW = RAW_DATA / 'stj_integras_metadata'
PROCESSED_DATA = DATA_ROOT / 'processed'
REPORTS_DATA = DATA_ROOT / 'reports' / 'summaries'
FIGURES_DATA = DATA_ROOT / 'reports' / 'figures'
REFERENCE_DATA = DATA_ROOT / 'reference'
API_DATA_CANDIDATES = [DATA_ROOT / 'api', PROJECT_ROOT / 'data' / 'api']
API_DATA = next((path for path in API_DATA_CANDIDATES if path.exists()), API_DATA_CANDIDATES[0])

for path in [PROCESSED_DATA, REPORTS_DATA, FIGURES_DATA, REFERENCE_DATA]:
    path.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('API_DATA:', API_DATA, 'exists=', API_DATA.exists())
print('RAW_STJ:', RAW_STJ, 'exists=', RAW_STJ.exists())
print('METADATA_RAW:', METADATA_RAW, 'exists=', METADATA_RAW.exists())
print('PROCESSED_DATA:', PROCESSED_DATA)

Google Drive ja esta montado.
PROJECT_ROOT: /content
DATA_ROOT: /content/drive/MyDrive/Mestrado/2026/llms/data
API_DATA: /content/drive/MyDrive/Mestrado/2026/llms/data/api exists= False
RAW_STJ: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras exists= True
METADATA_RAW: /content/drive/MyDrive/Mestrado/2026/llms/data/raw/stj_integras_metadata exists= True
PROCESSED_DATA: /content/drive/MyDrive/Mestrado/2026/llms/data/processed


In [3]:
# Ajuste o recorte temporal dos metadados de integras, quando disponiveis no Drive.
# Para exploracao, mantenha um recorte pequeno. Para uma rodada final completa, use None.
ANOS_ANALISE = [2026]  # None = todos os anos disponiveis
MAX_ARQUIVOS_METADATA = None  # ex.: 5 para teste rapido; None = sem limite
FORCAR_REPROCESSAR_METADATA = False
METADATA_CACHE_PATH = PROCESSED_DATA / 'stj_integras_documentos_manifest.parquet'

# Arquivos opcionais da API/dados abertos.
ATA_DISTRIBUICAO_PATH = API_DATA / 'ata20230630.json'
DICIONARIO_ATA_PATH = API_DATA / 'dicionario-atadedistribuicao.csv'
XSD_PATH = API_DATA / 'modelo-de-transferencia-de-dados-1.2-81544272558adf336e6c4d58ed66e4f7.xsd'

print('ATA_DISTRIBUICAO_PATH:', ATA_DISTRIBUICAO_PATH, ATA_DISTRIBUICAO_PATH.exists())
print('DICIONARIO_ATA_PATH:', DICIONARIO_ATA_PATH, DICIONARIO_ATA_PATH.exists())
print('XSD_PATH:', XSD_PATH, XSD_PATH.exists())

ATA_DISTRIBUICAO_PATH: /content/drive/MyDrive/Mestrado/2026/llms/data/api/ata20230630.json False
DICIONARIO_ATA_PATH: /content/drive/MyDrive/Mestrado/2026/llms/data/api/dicionario-atadedistribuicao.csv False
XSD_PATH: /content/drive/MyDrive/Mestrado/2026/llms/data/api/modelo-de-transferencia-de-dados-1.2-81544272558adf336e6c4d58ed66e4f7.xsd False


## 2. Funcoes de normalizacao

A chave principal para vida processual deve ser o numero CNJ de 20 digitos. O STJ tambem usa `numeroRegistro`, que e chave interna importante para atos e referencias do tribunal, mas nao substitui o numero unico CNJ.

In [4]:
def only_digits(value: Any) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    digits = re.sub(r'\D', '', str(value))
    return digits or None

def normalize_cnj(value: Any) -> str | None:
    digits = only_digits(value)
    if not digits:
        return None
    return digits if len(digits) == 20 else None

def normalize_process_text_key(value: Any) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = re.sub(r'\s+', ' ', str(value)).strip()
    if not text or text.lower() in {'nan', 'none', '<na>'}:
        return None
    return text

def parse_cnj(value: Any) -> dict[str, Any]:
    numero = normalize_cnj(value)
    if not numero:
        return {
            'numero_processo': None,
            'ano_origem_cnj': pd.NA,
            'segmento_cnj': pd.NA,
            'tribunal_cnj': pd.NA,
            'origem_cnj': pd.NA,
        }
    segment_labels = {
        '1': 'STF',
        '2': 'CNJ',
        '3': 'STJ',
        '4': 'Justica Federal',
        '5': 'Justica do Trabalho',
        '6': 'Justica Eleitoral',
        '7': 'Justica Militar da Uniao',
        '8': 'Justica Estadual',
        '9': 'Justica Militar Estadual',
    }
    return {
        'numero_processo': numero,
        'ano_origem_cnj': int(numero[9:13]),
        'segmento_cnj': segment_labels.get(numero[13], numero[13]),
        'tribunal_cnj': numero[14:16],
        'origem_cnj': numero[16:20],
    }

def parse_stj_datetime(value: Any) -> pd.Timestamp:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return pd.NaT
    text = str(value).strip()
    if not text:
        return pd.NaT
    if re.fullmatch(r'\d{14}', text):
        return pd.to_datetime(text, format='%Y%m%d%H%M%S', errors='coerce')
    if re.fullmatch(r'\d{8}', text):
        return pd.to_datetime(text, format='%Y%m%d', errors='coerce')
    return pd.to_datetime(text, errors='coerce', utc=False)

def first_non_null(values: pd.Series):
    clean = values.dropna()
    return clean.iloc[0] if not clean.empty else pd.NA

def join_unique(values: pd.Series, sep: str = ' | ') -> str:
    clean = values.dropna().astype(str).str.strip()
    clean = sorted(v for v in clean.unique() if v)
    return sep.join(clean)

def metadata_date_from_filename(filename: str) -> pd.Timestamp | None:
    match = re.fullmatch(r'metadados(\d{6}|\d{8})\.json', filename)
    if not match:
        return None
    raw = match.group(1)
    fmt = '%Y%m%d' if len(raw) == 8 else '%Y%m'
    return pd.to_datetime(raw, format=fmt, errors='coerce')

## 3. Ata de distribuicao do STJ

A ata descreve quando o processo foi registrado, distribuido, redistribuido ou atribuido no STJ. Ela e uma boa fonte para `data_entrada_stj` e relatoria inicial naquele evento.

In [5]:
def load_ata_distribuicao(path: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        print(f'Ata de distribuicao nao encontrada: {path}')
        empty = pd.DataFrame()
        return empty, empty, empty

    rows = json.loads(path.read_text(encoding='utf-8'))
    ata = pd.json_normalize(rows, sep='.')
    ata['numero_processo'] = ata['numeroUnico'].map(normalize_cnj)
    ata['numero_registro_stj'] = ata['numeroRegistro'].astype('string')
    ata['data_distribuicao_stj'] = ata['dataHoraDistribuicao'].map(parse_stj_datetime)
    ata['ano_origem_cnj'] = ata['numero_processo'].map(lambda v: parse_cnj(v)['ano_origem_cnj'] if v else pd.NA)

    partes_rows = []
    adv_rows = []
    for idx, item in enumerate(rows):
        numero = normalize_cnj(item.get('numeroUnico'))
        registro = item.get('numeroRegistro')
        for parte_idx, parte in enumerate(item.get('partes') or []):
            parte_row = {
                'numero_processo': numero,
                'numero_registro_stj': registro,
                'parte_idx': parte_idx,
                'descTipoParte': parte.get('descTipoParte'),
                'nomeParte': parte.get('nomeParte'),
                'numeroCNPJ': parte.get('numeroCNPJ'),
            }
            partes_rows.append(parte_row)
            for adv_idx, adv in enumerate(parte.get('advogados') or []):
                adv_rows.append({
                    'numero_processo': numero,
                    'numero_registro_stj': registro,
                    'parte_idx': parte_idx,
                    'advogado_idx': adv_idx,
                    'codigoOAB': adv.get('codigoOAB'),
                    'nomeAdvogado': adv.get('nomeAdvogado'),
                })

    partes = pd.DataFrame(partes_rows)
    advogados = pd.DataFrame(adv_rows)
    return ata, partes, advogados

ata_df, ata_partes, ata_advogados = load_ata_distribuicao(ATA_DISTRIBUICAO_PATH)
print('ata_df:', ata_df.shape)
print('ata_partes:', ata_partes.shape)
print('ata_advogados:', ata_advogados.shape)
ata_df.head()

Ata de distribuicao nao encontrada: /content/drive/MyDrive/Mestrado/2026/llms/data/api/ata20230630.json
ata_df: (0, 0)
ata_partes: (0, 0)
ata_advogados: (0, 0)


""


In [6]:
if not ata_df.empty:
    display(ata_df['descFormaDistribuicao'].value_counts().rename_axis('forma').reset_index(name='n'))
    display(ata_df['descDestino'].value_counts().head(20).rename_axis('destino').reset_index(name='n'))

## 4. DataJud: processos e movimentos

Os arquivos `stj_*_raw.json` do probe DataJud podem conter `movimentos`. Esta fonte e util para linha do tempo basica, mas nao deve ser tratada como corpus textual.

In [7]:
def iter_datajud_hits(raw_dir: Path):
    for path in sorted(raw_dir.glob('stj_*_raw.json')):
        try:
            data = json.loads(path.read_text(encoding='utf-8'))
        except Exception as exc:
            print(f'Erro lendo {path}: {exc}')
            continue
        hits = data.get('hits', {}).get('hits', []) if isinstance(data, dict) else []
        for hit in hits:
            src = hit.get('_source', {})
            yield path, src

def extract_datajud_tables(raw_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    proc_rows = []
    mov_rows = []
    assunto_rows = []

    for path, src in iter_datajud_hits(raw_dir):
        numero = normalize_cnj(src.get('numeroProcesso'))
        cnj = parse_cnj(numero)
        classe = src.get('classe') or {}
        orgao = src.get('orgaoJulgador') or {}
        proc_rows.append({
            **cnj,
            'fonte_datajud': path.name,
            'classe_codigo_datajud': classe.get('codigo') if isinstance(classe, dict) else pd.NA,
            'classe_nome_datajud': classe.get('nome') if isinstance(classe, dict) else pd.NA,
            'tribunal_datajud': src.get('tribunal'),
            'grau_datajud': src.get('grau'),
            'data_ajuizamento_datajud': parse_stj_datetime(src.get('dataAjuizamento')),
            'data_ultima_atualizacao_datajud': parse_stj_datetime(src.get('dataHoraUltimaAtualizacao')),
            'orgao_julgador_codigo_datajud': orgao.get('codigo') if isinstance(orgao, dict) else pd.NA,
            'orgao_julgador_nome_datajud': orgao.get('nome') if isinstance(orgao, dict) else pd.NA,
            'nivel_sigilo_datajud': src.get('nivelSigilo'),
            'n_movimentos_datajud': len(src.get('movimentos') or []),
        })

        for assunto in src.get('assuntos') or []:
            if isinstance(assunto, dict):
                assunto_rows.append({
                    'numero_processo': numero,
                    'assunto_codigo_datajud': assunto.get('codigo'),
                    'assunto_nome_datajud': assunto.get('nome'),
                })

        for idx, mov in enumerate(src.get('movimentos') or []):
            orgao_mov = mov.get('orgaoJulgador') or {}
            mov_rows.append({
                'numero_processo': numero,
                'movimento_idx_original': idx,
                'data_movimento': parse_stj_datetime(mov.get('dataHora')),
                'codigo_movimento': mov.get('codigo'),
                'nome_movimento': mov.get('nome'),
                'orgao_julgador_codigo_movimento': orgao_mov.get('codigo') if isinstance(orgao_mov, dict) else pd.NA,
                'orgao_julgador_nome_movimento': orgao_mov.get('nome') if isinstance(orgao_mov, dict) else pd.NA,
                'complementos_tabelados': json.dumps(mov.get('complementosTabelados'), ensure_ascii=False) if mov.get('complementosTabelados') is not None else None,
                'fonte_datajud': path.name,
            })

    processos = pd.DataFrame(proc_rows).drop_duplicates(subset=['numero_processo', 'fonte_datajud']) if proc_rows else pd.DataFrame()
    movimentos = pd.DataFrame(mov_rows)
    assuntos = pd.DataFrame(assunto_rows).drop_duplicates() if assunto_rows else pd.DataFrame()
    if not movimentos.empty:
        movimentos = movimentos.sort_values(['numero_processo', 'data_movimento', 'movimento_idx_original']).reset_index(drop=True)
    return processos, movimentos, assuntos

datajud_processos, datajud_movimentos, datajud_assuntos = extract_datajud_tables(RAW_DATA)
print('datajud_processos:', datajud_processos.shape)
print('datajud_movimentos:', datajud_movimentos.shape)
print('datajud_assuntos:', datajud_assuntos.shape)
datajud_processos.head()

datajud_processos: (0, 0)
datajud_movimentos: (0, 0)
datajud_assuntos: (0, 0)


""


In [8]:
if not datajud_movimentos.empty:
    display(datajud_movimentos['nome_movimento'].value_counts().head(30).rename_axis('movimento').reset_index(name='n'))
    display(datajud_movimentos.head(20))

## 5. Metadados das integras STJ

Esta etapa le os `metadados*.json` da base de integras. Eles representam documentos publicados/disponibilizados e permitem calcular a primeira aparicao do processo no corpus textual.

In [9]:
def metadata_sort_key(path: Path) -> str:
    match = re.fullmatch(r'metadados(\d{6}|\d{8})\.json', path.name)
    return match.group(1) if match else path.name

def find_metadata_files(metadata_root: Path, anos: list[int] | None = None) -> list[Path]:
    if not metadata_root.exists():
        print(f'METADATA_RAW nao existe: {metadata_root}')
        return []
    if anos is None:
        print('ANOS_ANALISE=None: buscando metadados recursivamente. Isto pode ser demorado em Drive grande.')
        files = list(metadata_root.rglob('metadados*.json'))
    else:
        files = []
        for ano in anos:
            year_dir = metadata_root / str(ano)
            files.extend(year_dir.glob('metadados*.json'))
            files.extend(metadata_root.glob(f'metadados{ano}*.json'))
            files.extend(metadata_root.glob(f'**/{ano}/metadados*.json'))
    files = sorted(set(files), key=metadata_sort_key)
    if MAX_ARQUIVOS_METADATA is not None:
        files = files[:MAX_ARQUIVOS_METADATA]
    return files

def load_integras_metadata(files: list[Path]) -> pd.DataFrame:
    frames = []
    for path in tqdm(files, desc='metadados'):
        try:
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                frame = pd.DataFrame(data)
            elif isinstance(data, dict):
                list_values = [v for v in data.values() if isinstance(v, list)]
                frame = pd.DataFrame(list_values[0]) if len(list_values) == 1 else pd.json_normalize(data)
            else:
                continue
            frame['arquivo_origem'] = path.name
            frame['data_arquivo_metadados'] = metadata_date_from_filename(path.name)
            frames.append(frame)
        except Exception as exc:
            print(f'Erro em {path}: {exc}')
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def filter_by_analysis_years(df: pd.DataFrame, anos: list[int] | None) -> pd.DataFrame:
    if df.empty or anos is None:
        return df
    mask = pd.Series(False, index=df.index)
    for col in ['data_documento', 'data_arquivo_metadados', 'dataPublicacao', 'dataDocumento', 'dataDisponibilizacao']:
        if col in df.columns:
            years = pd.to_datetime(df[col], errors='coerce').dt.year
            mask = mask | years.isin(anos)
    return df[mask].copy() if mask.any() else df

usar_cache_metadata = METADATA_CACHE_PATH.exists() and not FORCAR_REPROCESSAR_METADATA and MAX_ARQUIVOS_METADATA is None
if usar_cache_metadata:
    metadata_df = pd.DataFrame()
    integras_documentos_cache = pd.read_parquet(METADATA_CACHE_PATH)
    integras_documentos_cache = filter_by_analysis_years(integras_documentos_cache, ANOS_ANALISE)
    print('Manifesto de integras carregado do cache:', METADATA_CACHE_PATH)
    print('integras_documentos_cache:', integras_documentos_cache.shape)
else:
    integras_documentos_cache = pd.DataFrame()
    metadata_files = find_metadata_files(METADATA_RAW, ANOS_ANALISE)
    print('Arquivos de metadados encontrados:', len(metadata_files))
    if metadata_files:
        print('Primeiro arquivo:', metadata_files[0])
        print('Ultimo arquivo:', metadata_files[-1])
    metadata_df = load_integras_metadata(metadata_files)
    print('metadata_df:', metadata_df.shape)
metadata_df.head() if not metadata_df.empty else integras_documentos_cache.head()

Manifesto de integras carregado do cache: /content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_integras_documentos_manifest.parquet
integras_documentos_cache: (193979, 9)


,numero_processo,numero_registro_stj,seq_documento,data_documento,tipo_documento,ministro_documento,arquivo_origem,data_arquivo_metadados,assuntos_raw
0,None,202505108774,353144180,2026-01-02,DECISÃO,HERMAN BENJAMIN,metadados20260102.json,2026-01-02,12334
1,None,202505118087,353145378,2026-01-02,DECISÃO,HERMAN BENJAMIN,metadados20260102.json,2026-01-02,14227
2,None,202505107537,353132129,2026-01-02,DECISÃO,HERMAN BENJAMIN,metadados20260102.json,2026-01-02,3548
3,None,202505111903,353144178,2026-01-02,DECISÃO,HERMAN BENJAMIN,metadados20260102.json,2026-01-02,4355
4,None,202505101918,353132787,2026-01-02,DECISÃO,HERMAN BENJAMIN,metadados20260102.json,2026-01-02,4355


In [10]:
def pick_first_existing(columns: list[str], candidates: list[str]) -> str | None:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

def build_integras_documentos(metadata_df: pd.DataFrame) -> pd.DataFrame:
    if metadata_df.empty:
        return pd.DataFrame()

    processo_col = pick_first_existing(metadata_df.columns.tolist(), ['numeroProcesso', 'numeroUnico', 'processo', 'Processo'])
    registro_col = pick_first_existing(metadata_df.columns.tolist(), ['numeroRegistro', 'registro', 'numRegistro'])
    seq_col = pick_first_existing(metadata_df.columns.tolist(), ['SeqDocumento', 'seqDocumento', 'idDocumento'])
    data_col = pick_first_existing(metadata_df.columns.tolist(), ['dataPublicacao', 'dataDocumento', 'dataDisponibilizacao'])
    tipo_col = pick_first_existing(metadata_df.columns.tolist(), ['tipoDocumento', 'tipo', 'descricaoTipoDocumento'])
    ministro_col = pick_first_existing(metadata_df.columns.tolist(), ['ministro', 'NM_MINISTRO', 'relator'])

    docs = pd.DataFrame(index=metadata_df.index)
    docs['numero_processo'] = metadata_df[processo_col].map(normalize_cnj) if processo_col else pd.NA
    docs['numero_processo_original'] = metadata_df[processo_col].map(normalize_process_text_key) if processo_col else pd.NA
    docs['numero_registro_stj'] = metadata_df[registro_col].map(normalize_process_text_key) if registro_col else pd.NA
    docs['processo_agregacao'] = docs['numero_processo'].copy()
    docs['tipo_chave_processo'] = np.where(docs['numero_processo'].notna(), 'cnj', pd.NA)
    sem_cnj_com_registro = docs['processo_agregacao'].isna() & docs['numero_registro_stj'].notna()
    docs.loc[sem_cnj_com_registro, 'processo_agregacao'] = docs.loc[sem_cnj_com_registro, 'numero_registro_stj']
    docs.loc[sem_cnj_com_registro, 'tipo_chave_processo'] = 'registro_stj'
    sem_chave_com_original = docs['processo_agregacao'].isna() & docs['numero_processo_original'].notna()
    docs.loc[sem_chave_com_original, 'processo_agregacao'] = docs.loc[sem_chave_com_original, 'numero_processo_original']
    docs.loc[sem_chave_com_original, 'tipo_chave_processo'] = 'numero_original'
    docs['seq_documento'] = metadata_df[seq_col].astype('string').str.replace(r'\.0$', '', regex=True) if seq_col else pd.NA
    docs['data_documento'] = metadata_df[data_col].map(parse_stj_datetime) if data_col else pd.NaT
    docs['tipo_documento'] = metadata_df[tipo_col] if tipo_col else pd.NA
    docs['ministro_documento'] = metadata_df[ministro_col] if ministro_col else pd.NA
    docs['arquivo_origem'] = metadata_df.get('arquivo_origem', pd.Series(pd.NA, index=metadata_df.index))
    docs['data_arquivo_metadados'] = metadata_df.get('data_arquivo_metadados', pd.Series(pd.NaT, index=metadata_df.index))
    if 'assuntos' in metadata_df.columns:
        docs['assuntos_raw'] = metadata_df['assuntos']
    return docs

def ensure_process_keys(docs: pd.DataFrame) -> pd.DataFrame:
    if docs.empty:
        return docs
    docs = docs.copy()
    if 'numero_processo' not in docs.columns:
        docs['numero_processo'] = pd.NA
    if 'numero_processo_original' not in docs.columns:
        docs['numero_processo_original'] = pd.NA
    if 'numero_registro_stj' not in docs.columns:
        docs['numero_registro_stj'] = pd.NA
    if 'processo_agregacao' not in docs.columns:
        docs['processo_agregacao'] = docs['numero_processo'].copy()
        sem_cnj_com_registro = docs['processo_agregacao'].isna() & docs['numero_registro_stj'].notna()
        docs.loc[sem_cnj_com_registro, 'processo_agregacao'] = docs.loc[sem_cnj_com_registro, 'numero_registro_stj'].map(normalize_process_text_key)
        sem_chave_com_original = docs['processo_agregacao'].isna() & docs['numero_processo_original'].notna()
        docs.loc[sem_chave_com_original, 'processo_agregacao'] = docs.loc[sem_chave_com_original, 'numero_processo_original'].map(normalize_process_text_key)
    if 'tipo_chave_processo' not in docs.columns:
        docs['tipo_chave_processo'] = pd.NA
        docs.loc[docs['numero_processo'].notna(), 'tipo_chave_processo'] = 'cnj'
        docs.loc[docs['numero_processo'].isna() & docs['numero_registro_stj'].notna(), 'tipo_chave_processo'] = 'registro_stj'
        docs.loc[docs['tipo_chave_processo'].isna() & docs['numero_processo_original'].notna(), 'tipo_chave_processo'] = 'numero_original'
    return docs

if not integras_documentos_cache.empty:
    integras_documentos = ensure_process_keys(integras_documentos_cache)
else:
    integras_documentos = build_integras_documentos(metadata_df)
    integras_documentos = filter_by_analysis_years(integras_documentos, ANOS_ANALISE)
    integras_documentos = ensure_process_keys(integras_documentos)
print('integras_documentos:', integras_documentos.shape)
if not integras_documentos.empty:
    display(integras_documentos.head())
    print('Processos com numero CNJ:', integras_documentos['numero_processo'].notna().sum())
    print('Chaves auxiliares de processo:', integras_documentos['processo_agregacao'].nunique(dropna=True))
    display(integras_documentos['tipo_chave_processo'].value_counts(dropna=False).rename('documentos').reset_index())
    print('Documentos com seq_documento:', integras_documentos['seq_documento'].notna().sum())

integras_documentos: (193979, 12)


,numero_processo,numero_registro_stj,seq_documento,data_documento,tipo_documento,ministro_documento,arquivo_origem,data_arquivo_metadados,assuntos_raw,numero_processo_original,processo_agregacao,tipo_chave_processo
0,None,202505108774,353144180,2026-01-02,DECISÃO,HERMAN BENJAMIN,metadados20260102.json,2026-01-02,12334,<NA>,202505108774,registro_stj
1,None,202505118087,353145378,2026-01-02,DECISÃO,HERMAN BENJAMIN,metadados20260102.json,2026-01-02,14227,<NA>,202505118087,registro_stj
2,None,202505107537,353132129,2026-01-02,DECISÃO,HERMAN BENJAMIN,metadados20260102.json,2026-01-02,3548,<NA>,202505107537,registro_stj
3,None,202505111903,353144178,2026-01-02,DECISÃO,HERMAN BENJAMIN,metadados20260102.json,2026-01-02,4355,<NA>,202505111903,registro_stj
4,None,202505101918,353132787,2026-01-02,DECISÃO,HERMAN BENJAMIN,metadados20260102.json,2026-01-02,4355,<NA>,202505101918,registro_stj


Processos com numero CNJ: 0
Chaves auxiliares de processo: 180634


,tipo_chave_processo,documentos
0,registro_stj,193979


Documentos com seq_documento: 193979


## 6. Espinha dorsal por processo

A tabela abaixo tenta unir as fontes sem misturar granularidades: um processo pode ter varios documentos, varios movimentos e varias distribuicoes/redistribuicoes.

In [11]:
def summarize_ata(ata: pd.DataFrame) -> pd.DataFrame:
    if ata.empty:
        return pd.DataFrame(columns=['numero_processo'])
    return (
        ata.sort_values('data_distribuicao_stj')
        .groupby('numero_processo', dropna=True)
        .agg(
            numero_registro_stj=('numero_registro_stj', first_non_null),
            data_primeira_distribuicao_stj=('data_distribuicao_stj', 'min'),
            data_ultima_distribuicao_stj=('data_distribuicao_stj', 'max'),
            n_eventos_distribuicao_stj=('data_distribuicao_stj', 'size'),
            formas_distribuicao_stj=('descFormaDistribuicao', join_unique),
            classes_stj=('siglaClasse', join_unique),
            classe_cnj_codigos_stj=('codigoClasseCNJ', lambda s: join_unique(s.astype('string'))),
            relatores_stj=('nomeMinistroRelator', join_unique),
            destinos_stj=('descDestino', join_unique),
            assunto_cnj_ata=('codigoAssuntoCNJ', first_non_null),
        )
        .reset_index()
    )

def summarize_datajud(processos: pd.DataFrame, movimentos: pd.DataFrame, assuntos: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    if not processos.empty:
        proc = (
            processos.sort_values('data_ultima_atualizacao_datajud')
            .groupby('numero_processo', dropna=True)
            .agg(
                data_ajuizamento_datajud=('data_ajuizamento_datajud', 'min'),
                data_ultima_atualizacao_datajud=('data_ultima_atualizacao_datajud', 'max'),
                classe_nome_datajud=('classe_nome_datajud', first_non_null),
                classe_codigo_datajud=('classe_codigo_datajud', first_non_null),
                grau_datajud=('grau_datajud', first_non_null),
                orgao_julgador_nome_datajud=('orgao_julgador_nome_datajud', first_non_null),
                nivel_sigilo_datajud=('nivel_sigilo_datajud', first_non_null),
            )
            .reset_index()
        )
        pieces.append(proc)
    if not movimentos.empty:
        mov = (
            movimentos.groupby('numero_processo', dropna=True)
            .agg(
                data_primeiro_movimento_datajud=('data_movimento', 'min'),
                data_ultimo_movimento_datajud=('data_movimento', 'max'),
                n_movimentos_datajud=('data_movimento', 'size'),
                movimentos_datajud=('nome_movimento', join_unique),
            )
            .reset_index()
        )
        pieces.append(mov)
    if not assuntos.empty:
        ass = (
            assuntos.groupby('numero_processo', dropna=True)
            .agg(
                assuntos_datajud_codigos=('assunto_codigo_datajud', lambda s: join_unique(s.astype('string'))),
                assuntos_datajud_nomes=('assunto_nome_datajud', join_unique),
            )
            .reset_index()
        )
        pieces.append(ass)

    if not pieces:
        return pd.DataFrame(columns=['numero_processo'])
    result = pieces[0]
    for piece in pieces[1:]:
        result = result.merge(piece, on='numero_processo', how='outer')
    return result

def summarize_integras(docs: pd.DataFrame) -> pd.DataFrame:
    if docs.empty or 'numero_processo' not in docs.columns:
        return pd.DataFrame(columns=['numero_processo'])
    valid = docs.dropna(subset=['numero_processo']).copy()
    if valid.empty:
        return pd.DataFrame(columns=['numero_processo'])
    return (
        valid.groupby('numero_processo', dropna=True)
        .agg(
            primeira_aparicao_corpus=('data_documento', 'min'),
            ultima_aparicao_corpus=('data_documento', 'max'),
            n_documentos_corpus=('seq_documento', 'nunique'),
            tipos_documento_corpus=('tipo_documento', join_unique),
            ministros_documento_corpus=('ministro_documento', join_unique),
            arquivos_metadados_corpus=('arquivo_origem', lambda s: s.nunique()),
        )
        .reset_index()
    )

def summarize_integras_by_key(docs: pd.DataFrame) -> pd.DataFrame:
    if docs.empty or 'processo_agregacao' not in docs.columns:
        return pd.DataFrame(columns=['processo_agregacao'])
    valid = docs.dropna(subset=['processo_agregacao']).copy()
    if valid.empty:
        return pd.DataFrame(columns=['processo_agregacao'])
    return (
        valid.groupby('processo_agregacao', dropna=True)
        .agg(
            tipo_chave_processo=('tipo_chave_processo', first_non_null),
            numero_processo=('numero_processo', first_non_null),
            numero_registro_stj=('numero_registro_stj', first_non_null),
            numero_processo_original=('numero_processo_original', first_non_null),
            primeira_aparicao_corpus=('data_documento', 'min'),
            ultima_aparicao_corpus=('data_documento', 'max'),
            n_documentos_corpus=('seq_documento', 'nunique'),
            tipos_documento_corpus=('tipo_documento', join_unique),
            ministros_documento_corpus=('ministro_documento', join_unique),
            arquivos_metadados_corpus=('arquivo_origem', lambda s: s.nunique()),
        )
        .reset_index()
    )

ata_summary = summarize_ata(ata_df)
datajud_summary = summarize_datajud(datajud_processos, datajud_movimentos, datajud_assuntos)
integras_summary = summarize_integras(integras_documentos)
integras_summary_by_key = summarize_integras_by_key(integras_documentos)

print('ata_summary:', ata_summary.shape)
print('datajud_summary:', datajud_summary.shape)
print('integras_summary:', integras_summary.shape)
print('integras_summary_by_key:', integras_summary_by_key.shape)
integras_summary_by_key.head()

ata_summary: (0, 1)
datajud_summary: (0, 1)
integras_summary: (0, 1)
integras_summary_by_key: (180634, 11)


,processo_agregacao,tipo_chave_processo,numero_processo,numero_registro_stj,numero_processo_original,primeira_aparicao_corpus,ultima_aparicao_corpus,n_documentos_corpus,tipos_documento_corpus,ministros_documento_corpus,arquivos_metadados_corpus
0,200001149423,registro_stj,<NA>,200001149423,<NA>,2026-04-17,2026-04-17,1,ACÓRDÃO,GURGEL DE FARIA,1
1,200001267680,registro_stj,<NA>,200001267680,<NA>,2026-04-14,2026-04-14,1,DECISÃO,LUIS FELIPE SALOMÃO,1
2,200201173602,registro_stj,<NA>,200201173602,<NA>,2026-02-10,2026-02-10,1,DECISÃO,AFRÂNIO VILELA,1
3,200300067191,registro_stj,<NA>,200300067191,<NA>,2026-04-14,2026-04-14,1,DECISÃO,LUIS FELIPE SALOMÃO,1
4,200302046026,registro_stj,<NA>,200302046026,<NA>,2026-03-11,2026-03-11,1,DECISÃO,OG FERNANDES,1


In [12]:
process_ids = pd.Series(dtype='string')
for table in [ata_summary, datajud_summary, integras_summary]:
    if not table.empty and 'numero_processo' in table.columns:
        process_ids = pd.concat([process_ids, table['numero_processo'].dropna().astype('string')])
process_spine = pd.DataFrame({'numero_processo': sorted(process_ids.dropna().unique())})

for table in [ata_summary, datajud_summary, integras_summary]:
    if not table.empty:
        process_spine = process_spine.merge(table, on='numero_processo', how='left')

if not process_spine.empty:
    cnj_parts = process_spine['numero_processo'].map(parse_cnj).apply(pd.Series)
    for col in ['ano_origem_cnj', 'segmento_cnj', 'tribunal_cnj', 'origem_cnj']:
        process_spine[col] = cnj_parts[col]

    if 'primeira_aparicao_corpus' not in process_spine.columns:
        process_spine['primeira_aparicao_corpus'] = pd.NaT
    process_spine['primeira_aparicao_corpus'] = pd.to_datetime(process_spine['primeira_aparicao_corpus'], errors='coerce')

    process_spine['ja_existia_antes_da_primeira_aparicao_corpus'] = (
        process_spine['ano_origem_cnj'].notna()
        & process_spine['primeira_aparicao_corpus'].notna()
        & (process_spine['ano_origem_cnj'].astype('Int64') < process_spine['primeira_aparicao_corpus'].dt.year.astype('Int64'))
    )
    process_spine['ano_primeira_aparicao_corpus'] = process_spine['primeira_aparicao_corpus'].dt.year
    process_spine['anos_entre_origem_e_corpus'] = process_spine['ano_primeira_aparicao_corpus'] - process_spine['ano_origem_cnj'].astype('float')

print('process_spine:', process_spine.shape)
process_spine.head()

process_spine: (0, 1)


,numero_processo


## 7. Checks analiticos

Esses checks ajudam a verificar se estamos contando documentos, processos e eventos separadamente.

In [13]:
if not process_spine.empty:
    checks = {
        'processos_total': len(process_spine),
        'com_ata_distribuicao': int(process_spine['data_primeira_distribuicao_stj'].notna().sum()) if 'data_primeira_distribuicao_stj' in process_spine else 0,
        'com_datajud': int(process_spine['data_ultima_atualizacao_datajud'].notna().sum()) if 'data_ultima_atualizacao_datajud' in process_spine else 0,
        'com_documentos_corpus': int(process_spine['n_documentos_corpus'].fillna(0).gt(0).sum()) if 'n_documentos_corpus' in process_spine else 0,
        'ja_existia_antes_corpus': int(process_spine['ja_existia_antes_da_primeira_aparicao_corpus'].fillna(False).sum()) if 'ja_existia_antes_da_primeira_aparicao_corpus' in process_spine else 0,
    }
    checks_df = pd.Series(checks, name='valor').reset_index()
    checks_df.columns = ['metrica', 'valor']
    display(checks_df)

    if 'anos_entre_origem_e_corpus' in process_spine:
        display(process_spine['anos_entre_origem_e_corpus'].describe())
else:
    checks = {
        'processos_total_cnj': 0,
        'chaves_auxiliares_corpus': len(integras_summary_by_key),
        'documentos_manifesto': len(integras_documentos),
        'documentos_com_numero_cnj': int(integras_documentos['numero_processo'].notna().sum()) if 'numero_processo' in integras_documentos else 0,
    }
    checks_df = pd.Series(checks, name='valor').reset_index()
    checks_df.columns = ['metrica', 'valor']
    display(checks_df)

,metrica,valor
0,processos_total_cnj,0
1,chaves_auxiliares_corpus,180634
2,documentos_manifesto,193979
3,documentos_com_numero_cnj,0


In [14]:
if not process_spine.empty and 'ano_origem_cnj' in process_spine:
    display(
        process_spine.groupby('ano_origem_cnj', dropna=False)
        .size()
        .rename('processos')
        .reset_index()
        .sort_values('ano_origem_cnj')
        .tail(20)
    )

## 8. Salvar artefatos

Arquivos principais:

- `stj_processos_ciclo_vida.parquet`: uma linha por processo;
- `stj_movimentos_datajud.parquet`: uma linha por movimento;
- `stj_ata_distribuicoes.parquet`: uma linha por evento da ata;
- `stj_integras_documentos_manifest.parquet`: uma linha por documento/metadado textual, sem texto.

In [15]:
outputs = {}
if not process_spine.empty:
    path = PROCESSED_DATA / 'stj_processos_ciclo_vida.parquet'
    process_spine.to_parquet(path, index=False)
    outputs['processos'] = path
    process_spine.to_csv(PROCESSED_DATA / 'stj_processos_ciclo_vida.csv', index=False)

if not datajud_movimentos.empty:
    path = PROCESSED_DATA / 'stj_movimentos_datajud.parquet'
    datajud_movimentos.to_parquet(path, index=False)
    outputs['movimentos_datajud'] = path

if not ata_df.empty:
    path = PROCESSED_DATA / 'stj_ata_distribuicoes.parquet'
    ata_df.to_parquet(path, index=False)
    outputs['ata_distribuicoes'] = path

if not ata_partes.empty:
    path = PROCESSED_DATA / 'stj_ata_partes.parquet'
    ata_partes.to_parquet(path, index=False)
    outputs['ata_partes'] = path

if not integras_documentos.empty:
    path = PROCESSED_DATA / 'stj_integras_documentos_manifest.parquet'
    integras_documentos.to_parquet(path, index=False)
    outputs['documentos_manifest'] = path

if not integras_summary_by_key.empty:
    path = PROCESSED_DATA / 'stj_integras_corpus_por_chave.parquet'
    integras_summary_by_key.to_parquet(path, index=False)
    outputs['corpus_por_chave'] = path

print('Arquivos salvos:')
for name, path in outputs.items():
    print(f'- {name}: {path}')

Arquivos salvos:
- documentos_manifest: /content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_integras_documentos_manifest.parquet
- corpus_por_chave: /content/drive/MyDrive/Mestrado/2026/llms/data/processed/stj_integras_corpus_por_chave.parquet


In [16]:
summary_lines = [
    '# Ciclo de vida processual STJ',
    '',
    f'- Processos na espinha dorsal: {len(process_spine):,}',
    f'- Eventos de distribuicao/registro na ata: {len(ata_df):,}',
    f'- Movimentos DataJud: {len(datajud_movimentos):,}',
    f'- Documentos de integras no manifesto: {len(integras_documentos):,}',
    f'- Chaves auxiliares no corpus textual: {len(integras_summary_by_key):,}',
]
if not process_spine.empty and 'ja_existia_antes_da_primeira_aparicao_corpus' in process_spine:
    summary_lines.append(f'- Processos que ja existiam antes da primeira aparicao no corpus: {int(process_spine["ja_existia_antes_da_primeira_aparicao_corpus"].fillna(False).sum()):,}')
summary_lines.extend([
    '',
    '## Observacoes',
    '',
    '- DataJud/API ajuda a reconstruir metadados, movimentos e datas, mas nao deve ser assumido como fonte de texto integral.',
    '- A ata de distribuicao marca eventos no STJ; ela nao representa necessariamente o nascimento original do processo.',
    '- A primeira aparicao no corpus e uma propriedade da coleta de textos, nao do processo em si.',
])
report_path = REPORTS_DATA / 'stj_ciclo_vida_processual_summary.md'
report_path.write_text('\n'.join(summary_lines), encoding='utf-8')
report_path

PosixPath('/content/drive/MyDrive/Mestrado/2026/llms/data/reports/summaries/stj_ciclo_vida_processual_summary.md')

## 9. Proximos passos

Rode o notebook `07_documentos_por_processo_stj.ipynb` para anexar textos integrais aos documentos e produzir uma tabela longa documento-texto por processo.